# Text Analytics Assignment – Set D
### Q1: Word2Vec on Health & Fitness sentences  |  Q2: Text Preprocessing Pipeline

Both questions use small health & fitness text. Q1 trains word embeddings with Gensim,
Q2 builds a cleaning → tokenizing → stopwords → stemming → lemmatization pipeline with NLTK.

In [1]:
import re
import string

import nltk
import pandas as pd
from gensim.models import Word2Vec
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer, word_tokenize

# download nltk data only if it is not already there
needed = {
    "tokenizers/punkt": "punkt",
    "tokenizers/punkt_tab": "punkt_tab",
    "corpora/stopwords": "stopwords",
    "corpora/wordnet.zip": "wordnet",
    "corpora/omw-1.4.zip": "omw-1.4",
    "taggers/averaged_perceptron_tagger_eng": "averaged_perceptron_tagger_eng",
}
for path, pkg in needed.items():
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(pkg, quiet=True)

pd.set_option("display.max_colwidth", 80)

---
# Question 1 – Word2Vec on Health & Fitness Sentences

### Step 1: Load the sentences into a list

In [2]:
sentences = [
    "A balanced diet helps maintain a healthy body weight",
    "Regular workout sessions improve overall muscle strength",
    "Drinking enough water daily supports good digestion",
    "A high protein diet is common among strength athletes",
    "Cardio workouts help improve heart and lung health",
    "Skipping meals can slow down your metabolism over time",
    "Stretching before a workout reduces the risk of injury",
    "A balanced diet should include fruits, vegetables and whole grains",
    "Strength training workouts build muscle and improve bone density",
    "Getting enough sleep is important for muscle recovery",
    "A sudden change in diet can cause digestive discomfort",
    "Yoga is a low-impact workout that improves flexibility",
    "Doctors recommend a balanced diet to prevent lifestyle diseases",
    "Interval training workouts burn calories quickly",
    "A high-sugar diet is linked to several health problems",
    "Consistent workout routines improve long-term fitness levels",
    "Nutritionists often design a diet plan based on individual goals",
    "Warm-up exercises prepare the body for an intense workout",
    "A vegetarian diet can still provide sufficient protein",
    "Rest days are an important part of any workout schedule",
]

# saving it too, so the corpus is part of the submission
pd.DataFrame({"text": sentences}).to_csv("data/word2vec_corpus.csv", index=False)

print("Total sentences:", len(sentences))

Total sentences: 20


### Step 2 & 3: Lowercase, remove punctuation, tokenize

In [3]:
def clean_sentence(text):
    text = text.lower()
    # replace hyphen with space so 'low-impact' becomes 'low impact' instead of 'lowimpact'
    text = text.replace("-", " ")
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

cleaned = [clean_sentence(s) for s in sentences]
tokenized = [word_tokenize(s) for s in cleaned]

for s in tokenized[:3]:
    print(s)

['a', 'balanced', 'diet', 'helps', 'maintain', 'a', 'healthy', 'body', 'weight']
['regular', 'workout', 'sessions', 'improve', 'overall', 'muscle', 'strength']
['drinking', 'enough', 'water', 'daily', 'supports', 'good', 'digestion']


### Step 4 (optional): Stopword removal

I tried it both ways. Removing stopwords takes out words like *a, is, the* which otherwise show up
as "similar" to everything because they appear in almost every sentence. So I am training on the
stopword‑removed tokens.

In [4]:
stop_words = set(stopwords.words("english"))

tokens_no_stop = [[w for w in sent if w not in stop_words] for sent in tokenized]

vocab_with_stop = {w for sent in tokenized for w in sent}
vocab_without_stop = {w for sent in tokens_no_stop for w in sent}
print("Unique words WITH stopwords   :", len(vocab_with_stop))
print("Unique words WITHOUT stopwords:", len(vocab_without_stop))

Unique words WITH stopwords   : 122
Unique words WITHOUT stopwords: 102


### Step 5: Train the Word2Vec model

Dataset is tiny (20 sentences), so I kept the settings small:
- `vector_size=50` – small vectors are enough for ~100 words
- `window=3` – looks 3 words left and right
- `min_count=1` – keep every word, otherwise most words get dropped
- `sg=1` – skip-gram works a bit better than CBOW on small data
- `epochs=500` – many passes because there is very little data (with 200 epochs every word came out ~0.99 similar to every other word)
- `seed=42, workers=1` – so the result is repeatable

In [5]:
model = Word2Vec(
    sentences=tokens_no_stop,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=500,
    seed=42,
    workers=1,
)

print("Total sentences      :", len(tokens_no_stop))
print("Vocabulary size      :", len(model.wv.index_to_key))

Total sentences      : 20
Vocabulary size      : 102


### Step 6: Full vocabulary learned by the model

In [6]:
vocab = model.wv.index_to_key   # sorted by frequency, most common first
print(vocab)

['diet', 'workout', 'improve', 'workouts', 'strength', 'muscle', 'balanced', 'important', 'training', 'health', 'protein', 'high', 'enough', 'body', 'schedule', 'part', 'days', 'rest', 'sufficient', 'provide', 'still', 'vegetarian', 'intense', 'prepare', 'exercises', 'warm', 'goals', 'individual', 'based', 'plan', 'design', 'often', 'nutritionists', 'levels', 'fitness', 'term', 'long', 'routines', 'consistent', 'problems', 'several', 'linked', 'sugar', 'quickly', 'calories', 'burn', 'interval', 'diseases', 'lifestyle', 'prevent', 'recommend', 'doctors', 'flexibility', 'improves', 'impact', 'low', 'yoga', 'discomfort', 'digestive', 'cause', 'change', 'sudden', 'recovery', 'sleep', 'getting', 'density', 'bone', 'build', 'grains', 'whole', 'vegetables', 'fruits', 'include', 'injury', 'risk', 'reduces', 'stretching', 'time', 'metabolism', 'slow', 'meals', 'skipping', 'lung', 'heart', 'help', 'cardio', 'athletes', 'among', 'common', 'digestion', 'good', 'supports', 'daily', 'water', 'drinki

In [7]:
# same vocab with counts, easier to read as a table
vocab_df = pd.DataFrame({
    "word": vocab,
    "count": [model.wv.get_vecattr(w, "count") for w in vocab],
})
vocab_df.head(15)

,word,count
0,diet,8
1,workout,6
2,improve,4
3,workouts,3
4,strength,3
5,muscle,3
6,balanced,3
7,important,2
8,training,2
9,health,2


### Step 7: Word vector for \"diet\"

In [8]:
diet_vec = model.wv["diet"]
print("Shape of vector:", diet_vec.shape)
print("First 10 values:\n", diet_vec[:10].round(4))

Shape of vector: (50,)
First 10 values:
 [-0.2806 -0.1459  0.0633  0.1014 -0.078   0.1492  0.0044  0.3786 -0.1828
 -0.2185]


### Step 8 & 9: Top-5 similar words for \"diet\" and \"workout\"

In [9]:
def show_similar(word, topn=5):
    result = model.wv.most_similar(word, topn=topn)
    return pd.DataFrame(result, columns=["similar_word", "similarity"]).round(4)

print("Most similar to 'diet'")
show_similar("diet")

Most similar to 'diet'


,similar_word,similarity
0,digestive,0.9166
1,balanced,0.9096
2,sudden,0.9077
3,cause,0.9071
4,change,0.9069


In [10]:
print("Most similar to 'workout'")
show_similar("workout")

Most similar to 'workout'


,similar_word,similarity
0,consistent,0.9308
1,stretching,0.9253
2,levels,0.9179
3,injury,0.9157
4,reduces,0.9146


### Step 10: Interpretation

Partly yes. For *diet* the model gives *digestive, balanced, sudden, cause, change* – *balanced*
makes total sense ("balanced diet" appears 3 times) and the others all come from one sentence:
"a sudden change in diet can cause digestive discomfort". For *workout* it gives *consistent,
stretching, levels, injury, reduces*, which again are just the words from "stretching before a
workout reduces the risk of injury" and "consistent workout routines improve fitness levels".
So the model is not really learning meaning like "diet ≈ nutrition", it is mostly remembering
which words were in the same sentence. This happens because the model only learns from which words appear near each other, and
with only 20 sentences most words appear once or twice, so it does not get enough examples to tell
a real pattern from a coincidence. With a big corpus (millions of sentences) these random pairs get
averaged out and the similar words become much more meaningful.

### Interpretation Questions

**Why is `min_count` needed, and what if we increase it?**
`min_count` ignores words that appear fewer times than the given number. On real data this removes
typos and very rare words that the model cannot learn a good vector for anyway. In this dataset
almost every word appears only once, so if I set `min_count=2` or `3` the vocabulary shrinks a lot
(see the cell below) and words we care about might get dropped completely.

**What does increasing the window size change?**
The window is how many words on each side are treated as "context". A small window learns close,
almost grammatical relations (word that comes right before or after). A bigger window looks at the
whole sentence, so it learns more about the topic – words that are in the same kind of sentence
become similar even if they are not next to each other. In short sentences like ours, a big window
basically covers the full sentence.

**Why might results change if we retrain?**
Word2Vec starts with random vectors and also uses random negative sampling and random shuffling.
So every training run starts from a different point and ends at slightly different vectors. On a
small dataset this randomness matters more, so the top-5 list can change. Fixing `seed` and using
`workers=1` makes it repeatable.

In [11]:
# small experiment for the min_count question
for mc in [1, 2, 3]:
    m = Word2Vec(tokens_no_stop, vector_size=50, window=3, min_count=mc,
                 sg=1, epochs=500, seed=42, workers=1)
    print(f"min_count={mc} -> vocab size = {len(m.wv.index_to_key)}")

min_count=1 -> vocab size = 102


min_count=2 -> vocab size = 14


min_count=3 -> vocab size = 7


In [12]:
# small experiment for the retrain question - different seeds, different neighbours
for s in [1, 7, 99]:
    m = Word2Vec(tokens_no_stop, vector_size=50, window=3, min_count=1,
                 sg=1, epochs=500, seed=s, workers=1)
    print(f"seed={s}:", [w for w, _ in m.wv.most_similar("diet", topn=5)])

seed=1: ['protein', 'balanced', 'still', 'doctors', 'vegetarian']


seed=7: ['still', 'balanced', 'vegetarian', 'provide', 'sufficient']


seed=99: ['provide', 'protein', 'vegetarian', 'sudden', 'sufficient']


---
# Question 2 – Text Preprocessing Pipeline

### Step 1: Load raw sentences into a DataFrame (and save as dataset.csv)

In [13]:
raw_sentences = [
    "A balanced DIET helps maintain healthy body weight!!",
    "regular workout sessions improve muscle strength... love it :)",
    "drinking 8 glasses of water daily supports digestion!!",
    "skipping meals can SLOW down your metabolism...",
    "stretching before a workout reduces injury risk!! important",
    "strength training builds muscle & improves bone density",
    "getting 7-8 hrs of sleep is important for recovery :)",
    "yoga is a LOW-impact workout, improves flexibility!!",
    "a high-sugar diet is linked to health problems...",
    "interval training burns calories QUICKLY!! great results",
    "nutritionists design diet plans based on individual goals",
    "rest days are an important part of any workout schedule!!",
]

df = pd.DataFrame({"text": raw_sentences})
df.to_csv("data/dataset.csv", index=False)

df = pd.read_csv("data/dataset.csv")
df

,text
0,A balanced DIET helps maintain healthy body weight!!
1,regular workout sessions improve muscle strength... love it :)
2,drinking 8 glasses of water daily supports digestion!!
3,skipping meals can SLOW down your metabolism...
4,stretching before a workout reduces injury risk!! important
5,strength training builds muscle & improves bone density
6,getting 7-8 hrs of sleep is important for recovery :)
7,"yoga is a LOW-impact workout, improves flexibility!!"
8,a high-sugar diet is linked to health problems...
9,interval training burns calories QUICKLY!! great results


### Step 2: Cleaning function

In [14]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # drop punctuation, numbers, symbols
    text = re.sub(r"\s+", " ", text).strip() # collapse extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]]

,text,clean_text
0,A balanced DIET helps maintain healthy body weight!!,a balanced diet helps maintain healthy body weight
1,regular workout sessions improve muscle strength... love it :),regular workout sessions improve muscle strength love it
2,drinking 8 glasses of water daily supports digestion!!,drinking glasses of water daily supports digestion
3,skipping meals can SLOW down your metabolism...,skipping meals can slow down your metabolism
4,stretching before a workout reduces injury risk!! important,stretching before a workout reduces injury risk important
5,strength training builds muscle & improves bone density,strength training builds muscle improves bone density
6,getting 7-8 hrs of sleep is important for recovery :),getting hrs of sleep is important for recovery
7,"yoga is a LOW-impact workout, improves flexibility!!",yoga is a low impact workout improves flexibility
8,a high-sugar diet is linked to health problems...,a high sugar diet is linked to health problems
9,interval training burns calories QUICKLY!! great results,interval training burns calories quickly great results


### Step 3: Tokenization – `word_tokenize` vs `RegexpTokenizer`

To actually see a difference I also run both on the **raw** text, because on the cleaned text
there is no punctuation left, so both give the same result.

In [15]:
regex_tok = RegexpTokenizer(r"[a-zA-Z]+")

for i in range(3):
    print(f"Sentence {i+1}")
    print("  word_tokenize (clean)  :", word_tokenize(df.clean_text[i]))
    print("  RegexpTokenizer (clean):", regex_tok.tokenize(df.clean_text[i]))
    print("  word_tokenize (raw)    :", word_tokenize(df.text[i]))
    print("  RegexpTokenizer (raw)  :", regex_tok.tokenize(df.text[i]))
    print()

Sentence 1
  word_tokenize (clean)  : ['a', 'balanced', 'diet', 'helps', 'maintain', 'healthy', 'body', 'weight']
  RegexpTokenizer (clean): ['a', 'balanced', 'diet', 'helps', 'maintain', 'healthy', 'body', 'weight']
  word_tokenize (raw)    : ['A', 'balanced', 'DIET', 'helps', 'maintain', 'healthy', 'body', 'weight', '!', '!']
  RegexpTokenizer (raw)  : ['A', 'balanced', 'DIET', 'helps', 'maintain', 'healthy', 'body', 'weight']

Sentence 2
  word_tokenize (clean)  : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']
  RegexpTokenizer (clean): ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']
  word_tokenize (raw)    : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', '...', 'love', 'it', ':', ')']
  RegexpTokenizer (raw)  : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']

Sentence 3
  word_tokenize (clean)  : ['drinking', 'glasses', 'of', 'water', 'daily', 'supports', 'di

**Difference:** on cleaned text both outputs are identical. On raw text, `word_tokenize` keeps
punctuation as separate tokens (`!`, `...`, `:`, `)`), while `RegexpTokenizer` with `[a-zA-Z]+`
only keeps alphabetic words and throws everything else away. So I use the RegexpTokenizer tokens
for the rest of the pipeline.

In [16]:
df["tokens"] = df["clean_text"].apply(regex_tok.tokenize)
df[["clean_text", "tokens"]]

,clean_text,tokens
0,a balanced diet helps maintain healthy body weight,"[a, balanced, diet, helps, maintain, healthy, body, weight]"
1,regular workout sessions improve muscle strength love it,"[regular, workout, sessions, improve, muscle, strength, love, it]"
2,drinking glasses of water daily supports digestion,"[drinking, glasses, of, water, daily, supports, digestion]"
3,skipping meals can slow down your metabolism,"[skipping, meals, can, slow, down, your, metabolism]"
4,stretching before a workout reduces injury risk important,"[stretching, before, a, workout, reduces, injury, risk, important]"
5,strength training builds muscle improves bone density,"[strength, training, builds, muscle, improves, bone, density]"
6,getting hrs of sleep is important for recovery,"[getting, hrs, of, sleep, is, important, for, recovery]"
7,yoga is a low impact workout improves flexibility,"[yoga, is, a, low, impact, workout, improves, flexibility]"
8,a high sugar diet is linked to health problems,"[a, high, sugar, diet, is, linked, to, health, problems]"
9,interval training burns calories quickly great results,"[interval, training, burns, calories, quickly, great, results]"


### Step 4: Stopword removal

In [17]:
stop_words = set(stopwords.words("english"))

df["tokens_nostop"] = df["tokens"].apply(lambda toks: [w for w in toks if w not in stop_words])
df["count_before"] = df["tokens"].apply(len)
df["count_after"] = df["tokens_nostop"].apply(len)

df[["tokens_nostop", "count_before", "count_after"]]

,tokens_nostop,count_before,count_after
0,"[balanced, diet, helps, maintain, healthy, body, weight]",8,7
1,"[regular, workout, sessions, improve, muscle, strength, love]",8,7
2,"[drinking, glasses, water, daily, supports, digestion]",7,6
3,"[skipping, meals, slow, metabolism]",7,4
4,"[stretching, workout, reduces, injury, risk, important]",8,6
5,"[strength, training, builds, muscle, improves, bone, density]",7,7
6,"[getting, hrs, sleep, important, recovery]",8,5
7,"[yoga, low, impact, workout, improves, flexibility]",8,6
8,"[high, sugar, diet, linked, health, problems]",9,6
9,"[interval, training, burns, calories, quickly, great, results]",7,7


In [18]:
print("Total tokens before:", df.count_before.sum())
print("Total tokens after :", df.count_after.sum())

Total tokens before: 95
Total tokens after : 74


### Step 5: Porter Stemmer

In [19]:
porter = PorterStemmer()
df["porter"] = df["tokens_nostop"].apply(lambda toks: [porter.stem(w) for w in toks])
df[["tokens_nostop", "porter"]]

,tokens_nostop,porter
0,"[balanced, diet, helps, maintain, healthy, body, weight]","[balanc, diet, help, maintain, healthi, bodi, weight]"
1,"[regular, workout, sessions, improve, muscle, strength, love]","[regular, workout, session, improv, muscl, strength, love]"
2,"[drinking, glasses, water, daily, supports, digestion]","[drink, glass, water, daili, support, digest]"
3,"[skipping, meals, slow, metabolism]","[skip, meal, slow, metabol]"
4,"[stretching, workout, reduces, injury, risk, important]","[stretch, workout, reduc, injuri, risk, import]"
5,"[strength, training, builds, muscle, improves, bone, density]","[strength, train, build, muscl, improv, bone, densiti]"
6,"[getting, hrs, sleep, important, recovery]","[get, hr, sleep, import, recoveri]"
7,"[yoga, low, impact, workout, improves, flexibility]","[yoga, low, impact, workout, improv, flexibl]"
8,"[high, sugar, diet, linked, health, problems]","[high, sugar, diet, link, health, problem]"
9,"[interval, training, burns, calories, quickly, great, results]","[interv, train, burn, calori, quickli, great, result]"


### Step 6: Snowball Stemmer (English)

In [20]:
snowball = SnowballStemmer("english")
df["snowball"] = df["tokens_nostop"].apply(lambda toks: [snowball.stem(w) for w in toks])
df[["tokens_nostop", "snowball"]]

,tokens_nostop,snowball
0,"[balanced, diet, helps, maintain, healthy, body, weight]","[balanc, diet, help, maintain, healthi, bodi, weight]"
1,"[regular, workout, sessions, improve, muscle, strength, love]","[regular, workout, session, improv, muscl, strength, love]"
2,"[drinking, glasses, water, daily, supports, digestion]","[drink, glass, water, daili, support, digest]"
3,"[skipping, meals, slow, metabolism]","[skip, meal, slow, metabol]"
4,"[stretching, workout, reduces, injury, risk, important]","[stretch, workout, reduc, injuri, risk, import]"
5,"[strength, training, builds, muscle, improves, bone, density]","[strength, train, build, muscl, improv, bone, densiti]"
6,"[getting, hrs, sleep, important, recovery]","[get, hrs, sleep, import, recoveri]"
7,"[yoga, low, impact, workout, improves, flexibility]","[yoga, low, impact, workout, improv, flexibl]"
8,"[high, sugar, diet, linked, health, problems]","[high, sugar, diet, link, health, problem]"
9,"[interval, training, burns, calories, quickly, great, results]","[interv, train, burn, calori, quick, great, result]"


### Step 7: WordNet Lemmatizer (with POS tags)

Without a POS tag the lemmatizer treats every word as a noun, so *improves* stays *improves*.
Passing the POS tag fixes that.

In [21]:
lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize(tokens):
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(w, to_wordnet_pos(t)) for w, t in tagged]

df["lemma_no_pos"] = df["tokens_nostop"].apply(lambda toks: [lemmatizer.lemmatize(w) for w in toks])
df["lemma"] = df["tokens_nostop"].apply(lemmatize)
df[["tokens_nostop", "lemma_no_pos", "lemma"]]

,tokens_nostop,lemma_no_pos,lemma
0,"[balanced, diet, helps, maintain, healthy, body, weight]","[balanced, diet, help, maintain, healthy, body, weight]","[balance, diet, help, maintain, healthy, body, weight]"
1,"[regular, workout, sessions, improve, muscle, strength, love]","[regular, workout, session, improve, muscle, strength, love]","[regular, workout, session, improve, muscle, strength, love]"
2,"[drinking, glasses, water, daily, supports, digestion]","[drinking, glass, water, daily, support, digestion]","[drinking, glass, water, daily, support, digestion]"
3,"[skipping, meals, slow, metabolism]","[skipping, meal, slow, metabolism]","[skip, meal, slow, metabolism]"
4,"[stretching, workout, reduces, injury, risk, important]","[stretching, workout, reduces, injury, risk, important]","[stretch, workout, reduces, injury, risk, important]"
5,"[strength, training, builds, muscle, improves, bone, density]","[strength, training, build, muscle, improves, bone, density]","[strength, training, build, muscle, improve, bone, density]"
6,"[getting, hrs, sleep, important, recovery]","[getting, hr, sleep, important, recovery]","[get, hrs, sleep, important, recovery]"
7,"[yoga, low, impact, workout, improves, flexibility]","[yoga, low, impact, workout, improves, flexibility]","[yoga, low, impact, workout, improve, flexibility]"
8,"[high, sugar, diet, linked, health, problems]","[high, sugar, diet, linked, health, problem]","[high, sugar, diet, link, health, problem]"
9,"[interval, training, burns, calories, quickly, great, results]","[interval, training, burn, calorie, quickly, great, result]","[interval, training, burn, calorie, quickly, great, result]"


### Step 8: Comparison table – Original | Porter | Snowball | Lemma

Built for every distinct word in the dataset. Rows where Porter and Snowball disagree are
highlighted.

In [22]:
# map each word to its POS-aware lemma using the sentence context
lemma_map = {}
for toks, lems in zip(df.tokens_nostop, df.lemma):
    for w, l in zip(toks, lems):
        lemma_map.setdefault(w, l)

words = sorted(lemma_map)
compare = pd.DataFrame({
    "Original Word": words,
    "Porter Stem": [porter.stem(w) for w in words],
    "Snowball Stem": [snowball.stem(w) for w in words],
    "Lemma": [lemma_map[w] for w in words],
})
compare["Disagree"] = compare["Porter Stem"] != compare["Snowball Stem"]

def highlight(row):
    color = "background-color: #ffd6d6" if row["Disagree"] else ""
    return [color] * len(row)

print("Distinct words:", len(compare))
compare.style.apply(highlight, axis=1)

Distinct words: 63


,Original Word,Porter Stem,Snowball Stem,Lemma,Disagree
0,balanced,balanc,balanc,balance,False
1,based,base,base,base,False
2,body,bodi,bodi,body,False
3,bone,bone,bone,bone,False
4,builds,build,build,build,False
5,burns,burn,burn,burn,False
6,calories,calori,calori,calorie,False
7,daily,daili,daili,daily,False
8,days,day,day,day,False
9,density,densiti,densiti,density,False


In [23]:
# just the rows where the two stemmers disagree
compare[compare.Disagree]

,Original Word,Porter Stem,Snowball Stem,Lemma,Disagree
23,hrs,hr,hrs,hrs,True
42,quickly,quickli,quick,quickly,True


In [24]:
# a few extra words that are not in the dataset, just to show common disagreements
extra = ["generously", "fairly", "sportingly", "quickly", "lying"]
pd.DataFrame({
    "word": extra,
    "porter": [porter.stem(w) for w in extra],
    "snowball": [snowball.stem(w) for w in extra],
})

,word,porter,snowball
0,generously,gener,generous
1,fairly,fairli,fair
2,sportingly,sportingli,sport
3,quickly,quickli,quick
4,lying,lie,lie


### Step 9: Vocabulary size at each stage

In [25]:
def vocab_size(col):
    return len({w for toks in df[col] for w in toks})

stages = pd.DataFrame({
    "Stage": ["Raw tokens", "After stopword removal", "After Porter stemming",
              "After Snowball stemming", "After lemmatization"],
    "Vocabulary size": [vocab_size("tokens"), vocab_size("tokens_nostop"),
                        vocab_size("porter"), vocab_size("snowball"), vocab_size("lemma")],
})
stages

,Stage,Vocabulary size
0,Raw tokens,77
1,After stopword removal,63
2,After Porter stemming,62
3,After Snowball stemming,62
4,After lemmatization,62


### Step 10: Interpretation

Both stemming and lemmatization reduce the vocabulary because they merge different forms of the
same word (*workout/workouts*, *improve/improves*). Stemming just chops off endings using rules,
so it is fast but produces things like *balanc*, *digest*, *metabol*, *calori* which are not real
words. Lemmatization uses the WordNet dictionary and POS tags, so it gives proper words like
*improve*, *build*, *burn*, but it is slower and needs the POS tag to work well. For a search
engine I would pick stemming, because it is fast and grouping more word forms together helps
recall, and nobody sees the stems anyway. For classification or anything where the output words are
shown to a person (topic labels, word clouds, explanations), I would prefer lemmatization because
the tokens stay readable and meaningful.

### Interpretation Questions

**Why can stemming give non-dictionary words while lemmatization always gives a valid word?**
A stemmer is just a set of suffix rules – for example "remove *-ing*", "change *-ies* to *-i*". It
does not check if the result is a real word, so *calories* becomes *calori* and *density* becomes
*densiti*. A lemmatizer looks the word up in a dictionary (WordNet) and returns its base form
(lemma), so the output is always an actual word.

**RegexpTokenizer vs word_tokenize – practical difference?**
`word_tokenize` is a trained, rule-based tokenizer that understands English: it splits punctuation
into their own tokens and handles contractions (*don't* → *do*, *n't*). `RegexpTokenizer` simply
returns whatever matches the pattern I give it, so with `[a-zA-Z]+` it keeps only letters.
I would use `word_tokenize` when punctuation matters (sentiment – "!!" and ":)" carry meaning,
or parsing/POS tagging), and `RegexpTokenizer` when I just want clean words quickly and want full
control over what counts as a token.

**A word where Porter and Snowball differ, and why.**
On this dataset two words differ: *quickly* → Porter `quickli`, Snowball `quick`, and *hrs* →
Porter `hr`, Snowball `hrs`. Snowball is
basically "Porter 2" – the same author (Martin Porter) improved the original algorithm. It has
updated rules, for example it has a special rule for *-ly* adverb endings, handles the final *-y* differently, and
leaves very short words alone (that is why *hrs* is not touched). Because the rule lists are different, the same word can
end up with a different stem.